In [70]:
import warnings
warnings.filterwarnings("ignore")

In [71]:
import os
import torch
import numpy as np
import pandas as pd

from flonacomldft.utils.io_utils import (
    load_pickle_file,
    get_project_path
)

from flonacomldft.internal_coordinates import (
    Coordinates_mapping
)

from flonacomldft.utils.diagnostics import R_hat

In [72]:
methods = ['adaptive', 'mixture_model']

In [73]:
simulation_types = dict(zip( methods, [
                    ['ab-flowMC', 'ab-flowMC w.o. MLP'],
                    ['ab-flowMM', 'ab-flowMM w.o. MLP']
                ])
            )

In [74]:
simulation_types

{'adaptive': ['ab-flowMC', 'ab-flowMC w.o. MLP'],
 'mixture_model': ['ab-flowMM', 'ab-flowMM w.o. MLP']}

In [75]:
simulation_idxs = { methods[0]: { simulation_types[methods[0]][0]: {
                        0: 31959916,
                        1: 31959919
                    },

                    simulation_types[methods[0]][1]: {
                        0: 31961552,
                        1: 31961553
                    }
                },

                methods[1]: { 
                    simulation_types[methods[1]][0]: 
                        32094541,
                    simulation_types[methods[1]][1]:
                        32094594
                        
                }
            }

In [76]:
simulation_idxs

{'adaptive': {'ab-flowMC': {0: 31959916, 1: 31959919},
  'ab-flowMC w.o. MLP': {0: 31961552, 1: 31961553}},
 'mixture_model': {'ab-flowMM': 32094541, 'ab-flowMM w.o. MLP': 32094594}}

In [77]:
simulations_folders = {
    
    methods[0]: {
    simulation_types[methods[0]][0]: get_project_path() + "/2-adaptive-mlp",
    simulation_types[methods[0]][1]: get_project_path() + "/1-adaptive"
    },

    methods[1]: {
    simulation_types[methods[1]][0]: get_project_path() + "/5-multimodal-mlp",
    simulation_types[methods[1]][1]: get_project_path() + "/4-multimodal"
    }
}

save_path = get_project_path() + "/figures/preprint_v2/data"

In [78]:
simulations_folders

{'adaptive': {'ab-flowMC': '/mnt/home/amolina/ceph/adaptive-flow-mc/2-adaptive-mlp',
  'ab-flowMC w.o. MLP': '/mnt/home/amolina/ceph/adaptive-flow-mc/1-adaptive'},
 'mixture_model': {'ab-flowMM': '/mnt/home/amolina/ceph/adaptive-flow-mc/5-multimodal-mlp',
  'ab-flowMM w.o. MLP': '/mnt/home/amolina/ceph/adaptive-flow-mc/4-multimodal'}}

In [79]:
get_simulation_filename_adaptive = lambda isomer_label, idx: "results_adaptive_is{:d}_{:d}/adaptive_sampling_is{:d}_{:d}.pkl".format(
    isomer_label, idx, isomer_label, idx)
get_cvs_filename_adaptive = lambda isomer_label, idx: "results_adaptive_is{:d}_{:d}/cvs_is{:d}_{:d}.pkl".format(
    isomer_label, idx, isomer_label, idx)

In [80]:
get_simulation_filename_mm = lambda idx: "results_multimodal_sampling_mixture_{:d}/checkpoint_NA_3000.pkl".format(
    idx)
get_cvs_filename_mm = lambda idx: "cvs_temporal/cvs_slices_{:d}_3000.pkl".format(
    idx)

In [81]:
simulations_folders[methods[0]]

{'ab-flowMC': '/mnt/home/amolina/ceph/adaptive-flow-mc/2-adaptive-mlp',
 'ab-flowMC w.o. MLP': '/mnt/home/amolina/ceph/adaptive-flow-mc/1-adaptive'}

# 1. Ab-flowMC simulations

In [82]:
method = methods[0]

adaptives = {}
cvss = {}

print("Loading data...")
for simulation in simulation_types[method]:
    adaptives[simulation] = {}
    cvss[simulation] = {}
    for key, idx in simulation_idxs[method][simulation].items():
        adaptives[simulation][key] = load_pickle_file(get_simulation_filename_adaptive(key, idx), 
        simulations_folders[method][simulation])
        cvss[simulation][key] = load_pickle_file(get_cvs_filename_adaptive(key, idx),
        simulations_folders[method][simulation])
        print("Loaded adaptive sampling {:s} and cvs for isomer {:d} and idx {:d}".format(
            simulation, key, idx))

Loading data...
Loaded adaptive sampling ab-flowMC and cvs for isomer 0 and idx 31959916


Loaded adaptive sampling ab-flowMC and cvs for isomer 1 and idx 31959919
Loaded adaptive sampling ab-flowMC w.o. MLP and cvs for isomer 0 and idx 31961552
Loaded adaptive sampling ab-flowMC w.o. MLP and cvs for isomer 1 and idx 31961553


In [83]:
adaptive = adaptives[simulation_types[method][0]][0]

## 1.a Acceptance rate

In [84]:
accs_df = []

for simulation in simulation_types[method]:
    
    for key, idx in simulation_idxs[method][simulation].items():

        df = pd.DataFrame(columns=['Method', 'Isomer', 'Step', 'Acceptance Rate'])
        accs = torch.cat(adaptives[simulation][key]['accs']).float().mean(dim=1).numpy()

        df['Acceptance Rate'] = accs
        df['Isomer'] = key
        df['Step'] = np.arange(0, len(accs))
        df['Method'] = simulation

        accs_df.append(df)

accs_df = pd.concat(accs_df)
accs_df.to_csv(save_path + "/acceptance_rates.csv", index=False)
print("Acceptance rates saved to {:s}".format(save_path + "/acceptance_rates.csv"))

Acceptance rates saved to /mnt/home/amolina/ceph/adaptive-flow-mc/figures/preprint_v2/data/acceptance_rates.csv


## 1.a Collective variables and potential energy

In [85]:
cvs_us_df = []

for simulation in simulation_types[method]:
    for isomer_label in [0, 1]:

        us = torch.cat(adaptives[simulation][isomer_label]['us']).reshape(1000, 50, 1)
        cvs = cvss[simulation][isomer_label]

        cvs_us = torch.cat([us, cvs], dim=2).numpy().reshape(50000, 3)

        df = pd.DataFrame(cvs_us, columns=['U', 'C', 'R'])
        df['Isomer'] = isomer_label
        df['Method'] = simulation

        cvs_us_df.append(df)

cvs_us_df = pd.concat(cvs_us_df)
cvs_us_df.to_csv(save_path + "/cvs_potential_energy.csv", index=False)
print("CVs and potential energy saved to {:s}".format(save_path + "/cvs_potential_energy.csv"))
cvs_us_df

CVs and potential energy saved to /mnt/home/amolina/ceph/adaptive-flow-mc/figures/preprint_v2/data/cvs_potential_energy.csv


,U,C,R,Isomer,Method
0,-6.773585,10.105214,2.457276,0,ab-flowMC
1,-6.750111,10.205934,2.442652,0,ab-flowMC
2,-6.764058,9.652035,2.473082,0,ab-flowMC
3,-6.732985,10.172332,2.427748,0,ab-flowMC
4,-6.748763,9.293897,2.517407,0,ab-flowMC
...,...,...,...,...,...
49995,-6.704685,10.828248,2.220704,1,ab-flowMC w.o. MLP
49996,-6.746437,10.425041,2.232781,1,ab-flowMC w.o. MLP
49997,-6.714196,11.065727,2.218863,1,ab-flowMC w.o. MLP
49998,-6.743561,11.145951,2.213711,1,ab-flowMC w.o. MLP


## 1.c Rhat

In [86]:
rhat_df = []

for simulation in simulation_types[method]:
    for isomer_label in [0, 1]:

        us = torch.cat(adaptives[simulation][isomer_label]['us']).reshape(1000, 50, 1)
        cvs = cvss[simulation][isomer_label]
        
        time_slices = adaptives[simulation][isomer_label]['time_mcmc']
        time_flatten = [t for time_slice in time_slices for t in time_slice]
        time_min = min(time_flatten)
        time_mcmc = np.array([t - time_min for t in time_flatten])/3600

        rhat_info = torch.cat((cvs, us), dim=2).detach().numpy().transpose(1, 0, 2)

        rhat = R_hat(rhat_info, labels=['C', 'R', 'U'])

        df = pd.DataFrame(rhat[0], columns=['C', 'R', 'U'])
        df['Time (h)'] = time_mcmc[rhat[1]]
        df['MCMC step'] = rhat[1]
        df['Method'] = simulation
        df['Isomer'] = isomer_label

        rhat_df.append(df)

rhat_df = pd.concat(rhat_df)
rhat_df.to_csv(save_path + "/rhat.csv", index=False)        
rhat_df

,C,R,U,Time (h),MCMC step,Method,Isomer
0,1.210686,1.228267,1.194109,2.800454,90,ab-flowMC,0
1,1.121994,1.119943,1.107089,5.679586,180,ab-flowMC,0
2,1.065124,1.070826,1.065830,8.531614,270,ab-flowMC,0
3,1.042406,1.042423,1.055567,11.332435,360,ab-flowMC,0
4,1.029414,1.032330,1.040203,14.137912,450,ab-flowMC,0
5,1.028392,1.029847,1.033139,16.935509,540,ab-flowMC,0
6,1.021719,1.023650,1.026424,19.749465,630,ab-flowMC,0
7,1.018697,1.021742,1.021554,22.615388,720,ab-flowMC,0
8,1.017255,1.021114,1.017018,25.403506,810,ab-flowMC,0
9,1.016924,1.018331,1.016177,28.190027,900,ab-flowMC,0


## 1.d MLP datasets

In [87]:
mlp_datasets_df = []

simulation = simulation_types[method][0]

for isomer_label in [0, 1]:
    datasets = adaptives[simulation][isomer_label]['mlps_datasets']
    models = adaptives[simulation][isomer_label]['dict_mlps'][-1][0]['model']
    for key in ['train', 'test']:
        df = pd.DataFrame(columns=['Predictions', 'True values'])
    
        data = datasets[key][0].detach().numpy()
        predict = models(torch.tensor(data[:, :12]).float()).detach().numpy()

        df['True values'] = data[:, 12]
        df['Predictions'] = predict
        df['Isomer'] = isomer_label
        df['Dataset'] = key.capitalize()
        df['Method'] = simulation

        mlp_datasets_df.append(df)

mlp_datasets_df = pd.concat(mlp_datasets_df)
mlp_datasets_df.to_csv(save_path + "/mlp_datasets.csv", index=False)
print("MLP datasets saved to {:s}".format(save_path + "/mlp_datasets.csv"))


MLP datasets saved to /mnt/home/amolina/ceph/adaptive-flow-mc/figures/preprint_v2/data/mlp_datasets.csv


## 1.d Optical spectrum

# 2. Ab-flowMM

In [88]:
method, simulation_idxs[method].items()

('adaptive',
 dict_items([('ab-flowMC', {0: 31959916, 1: 31959919}), ('ab-flowMC w.o. MLP', {0: 31961552, 1: 31961553})]))

In [89]:
method = methods[1]

mixture_models = {}
cvs_mixture_models = {}

print("Loading data...")
for key, idx in simulation_idxs[method].items():
    print(key, idx)
    mixture_models[key] = load_pickle_file(get_simulation_filename_mm(idx), 
    simulations_folders[method][key])
    cvs_mixture_models[key] = load_pickle_file(get_cvs_filename_mm(idx),
    get_project_path())
    print("Loaded mixture model sampling {:s} and cvs and idx {:d}".format(
        key, idx))

Loading data...
ab-flowMM 32094541
Loaded mixture model sampling ab-flowMM and cvs and idx 32094541
ab-flowMM w.o. MLP 32094594
Loaded mixture model sampling ab-flowMM w.o. MLP and cvs and idx 32094594


## 2a. Acceptace rate

In [90]:
mixture_models.keys()

dict_keys(['ab-flowMM', 'ab-flowMM w.o. MLP'])

In [91]:
accs_df_mm = []

    
for key in mixture_models.keys():

    df = pd.DataFrame(columns=['Method', 'Isomer', 'Step', 'Acceptance Rate'])
    accs = mixture_models[key]['accs'].float().mean(dim=1).numpy()

    df['Acceptance Rate'] = accs
    df['Isomer'] = key
    df['Step'] = np.arange(0, len(accs))
    df['Method'] = key

    accs_df_mm.append(df)

accs_df_mm = pd.concat(accs_df_mm)
accs_df_mm.to_csv(save_path + "/acceptance_rates_mm.csv", index=False)
print("Acceptance rates saved to {:s}".format(save_path + "/acceptance_rates_mm.csv"))

Acceptance rates saved to /mnt/home/amolina/ceph/adaptive-flow-mc/figures/preprint_v2/data/acceptance_rates_mm.csv


In [92]:
accs_df_mm

,Method,Isomer,Step,Acceptance Rate
0,ab-flowMM,ab-flowMM,0,0.45
1,ab-flowMM,ab-flowMM,1,0.25
2,ab-flowMM,ab-flowMM,2,0.30
3,ab-flowMM,ab-flowMM,3,0.40
4,ab-flowMM,ab-flowMM,4,0.30
...,...,...,...,...
2996,ab-flowMM w.o. MLP,ab-flowMM w.o. MLP,2996,0.25
2997,ab-flowMM w.o. MLP,ab-flowMM w.o. MLP,2997,0.30
2998,ab-flowMM w.o. MLP,ab-flowMM w.o. MLP,2998,0.50
2999,ab-flowMM w.o. MLP,ab-flowMM w.o. MLP,2999,0.45


## 2.b Collective variables

In [94]:
cvs_us_df_mm = []

for key in mixture_models.keys():

    us = mixture_models[key]['us'].reshape(3001, 20, 1)
    isomers = mixture_models[key]['isomers'].reshape(3001, 20, 1)
    cvs = cvs_mixture_models[key]
    cvs_us = torch.cat([us, cvs, isomers], dim=2).numpy().reshape(60020, 4)
    df = pd.DataFrame(cvs_us, columns=['U', 'C', 'R', 'Isomer'])
    df['Method'] = key
    cvs_us_df_mm.append(df)

cvs_us_df_mm = pd.concat(cvs_us_df_mm)
cvs_us_df_mm.to_csv(save_path + "/cvs_potential_energy_mm.csv", index=False)
print("CVs and potential energy saved to {:s}".format(save_path + "/cvs_potential_energy.csv"))
cvs_us_df

CVs and potential energy saved to /mnt/home/amolina/ceph/adaptive-flow-mc/figures/preprint_v2/data/cvs_potential_energy.csv


,U,C,R,Isomer,Method
0,-6.773585,10.105214,2.457276,0,ab-flowMC
1,-6.750111,10.205934,2.442652,0,ab-flowMC
2,-6.764058,9.652035,2.473082,0,ab-flowMC
3,-6.732985,10.172332,2.427748,0,ab-flowMC
4,-6.748763,9.293897,2.517407,0,ab-flowMC
...,...,...,...,...,...
49995,-6.704685,10.828248,2.220704,1,ab-flowMC w.o. MLP
49996,-6.746437,10.425041,2.232781,1,ab-flowMC w.o. MLP
49997,-6.714196,11.065727,2.218863,1,ab-flowMC w.o. MLP
49998,-6.743561,11.145951,2.213711,1,ab-flowMC w.o. MLP


## 2.c Rhat

In [95]:
mixture_models['ab-flowMM']['time_mcmc']

[1708189151.0943224,
 1708189174.2343788,
 1708189195.588432,
 1708189214.7881203,
 1708189234.200983,
 1708189258.52606,
 1708189281.175341,
 1708189303.967947,
 1708189327.502511,
 1708189349.9778516,
 1708189368.9729922,
 1708189390.6472232,
 1708189412.9873347,
 1708189436.6245706,
 1708189462.4731402,
 1708189485.3448074,
 1708189508.380226,
 1708189532.071178,
 1708189553.3794088,
 1708189576.5040863,
 1708189594.4745004,
 1708189619.405209,
 1708189640.1275144,
 1708189661.7312138,
 1708189689.228767,
 1708189709.1145444,
 1708189734.914357,
 1708189753.793246,
 1708189774.7511375,
 1708189801.594333,
 1708189819.8925445,
 1708189841.774198,
 1708189864.8293457,
 1708189883.103205,
 1708189906.264728,
 1708189930.2061539,
 1708189948.5213995,
 1708189967.9747782,
 1708189992.6623697,
 1708190010.445664,
 1708190034.2037446,
 1708190055.3663304,
 1708190076.0757465,
 1708190093.7077901,
 1708190113.6783462,
 1708190138.511592,
 1708190157.0238364,
 1708190185.494059,
 1708190204.

In [97]:
rhat_df_mm = []

for key in mixture_models.keys():

    us = mixture_models[key]['us'].reshape(3001, 20, 1)
    cvs = cvs_mixture_models[key]
    
    time_mm = mixture_models[key]['time_mcmc']
    time_min = min(time_mm)
    time_mcmc = np.array([t - time_min for t in time_mm])/3600

    rhat_info = torch.cat((cvs, us), dim=2).detach().numpy().transpose(1, 0, 2)

    rhat = R_hat(rhat_info, labels=['C', 'R', 'U'])

    df = pd.DataFrame(rhat[0], columns=['C', 'R', 'U'])
    df['Time (h)'] = time_mcmc[rhat[1]]
    df['MCMC step'] = rhat[1]
    df['Method'] = key

    rhat_df_mm.append(df)

rhat_df_mm = pd.concat(rhat_df_mm)
rhat_df_mm.to_csv(save_path + "/rhat.csv", index=False)        
rhat_df_mm

,C,R,U,Time (h),MCMC step,Method
0,1.244847,1.203371,1.201282,1.728772,290,ab-flowMM
1,1.221467,1.188414,1.174443,3.484163,580,ab-flowMM
2,1.130501,1.171119,1.089868,5.252929,870,ab-flowMM
3,1.101117,1.171875,1.096969,7.032640,1160,ab-flowMM
4,1.085846,1.177203,1.099640,8.837655,1450,ab-flowMM
5,1.085960,1.175499,1.074861,10.671688,1740,ab-flowMM
6,1.080660,1.156738,1.070799,12.524613,2030,ab-flowMM
7,1.081210,1.160547,1.075155,14.406494,2320,ab-flowMM
8,1.079550,1.164322,1.074507,16.304235,2610,ab-flowMM
9,1.078325,1.156593,1.064598,18.301156,2901,ab-flowMM


# 2.d Isomers

In [101]:
isomers_jumps = []

for key in mixture_models.keys():
    
    isomers = mixture_models[key]['isomers'].int()#.reshape(3001, 20, 1)

    df = pd.DataFrame(isomers, columns=['chain {:d}'.format(i) for i in np.arange(20)])
    df['Step'] = np.arange(0, 3001)
    df['Method'] = key
    
    isomers_jumps.append(df)

isomers_jumps = pd.concat(isomers_jumps)
isomers_jumps.to_csv(save_path + "/isomers_jumps_mm.csv", index=False)
print("Isomers jumps saved to {:s}".format(save_path + "/isomers_jumps_mm.csv"))

## 2.e Populations